In [6]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import argparse
import os
from typing import Tuple, Dict

import matplotlib.pyplot as plt
import numpy as np
from Bio import SeqIO
from Bio.Align import PairwiseAligner, substitution_matrices


# ---------- utils ----------

def read_one_fasta(path: str) -> Tuple[str, str]:
    """Прочитать первую запись из FASTA: вернуть (id, seq_upper)."""
    recs = list(SeqIO.parse(path, "fasta"))
    if not recs:
        raise ValueError(f"No sequences in {path}")
    return recs[0].id, str(recs[0].seq).upper()


def build_gapped_strings(seq1: str, seq2: str, alignment) -> Tuple[str, str]:
    """
    Восстановить выровненные строки с '-' по координатам пути (Alignment.coordinates).
    Работает для PairwiseAligner (Biopython 1.85+).
    """
    coords = np.asarray(alignment.coordinates, dtype=int)  # shape (2, m)
    a1, a2 = [], []
    for k in range(1, coords.shape[1]):
        i0, j0 = coords[0, k - 1], coords[1, k - 1]
        i1, j1 = coords[0, k],     coords[1, k]
        di, dj = i1 - i0, j1 - j0

        if di > 0 and dj > 0:
            # сопоставленный блок (match/mismatch)
            a1.append(seq1[i0:i1])
            a2.append(seq2[j0:j1])
        elif di > 0 and dj == 0:
            # разрыв во 2-й последовательности (del)
            a1.append(seq1[i0:i1])
            a2.append("-" * di)
        elif di == 0 and dj > 0:
            # разрыв в 1-й последовательности (ins)
            a1.append("-" * dj)
            a2.append(seq2[j0:j1])
        # di == dj == 0 — не бывает между соседними узлами
    return "".join(a1), "".join(a2)


def calc_metrics(a1: str, a2: str, score: float) -> Dict[str, float]:
    """Посчитать length, matches, identity, gaps — по гаппированным строкам."""
    assert len(a1) == len(a2)
    L = len(a1)
    matches = sum(1 for x, y in zip(a1, a2) if x == y and x != "-" and y != "-")
    no_gaps_len = sum(1 for x, y in zip(a1, a2) if x != "-" and y != "-")
    identity = (matches / no_gaps_len * 100) if no_gaps_len else 0.0
    gaps1 = a1.count("-")
    gaps2 = a2.count("-")
    return {
        "length_aligned": L,
        "matches": matches,
        "no_gaps_len": no_gaps_len,
        "identity": identity,
        "gaps_seq1": gaps1,
        "gaps_seq2": gaps2,
        "score": float(score),
    }


def save_txt(out_txt: str, id1: str, id2: str, a1: str, a2: str, m: Dict[str, float]) -> None:
    os.makedirs(os.path.dirname(out_txt) or ".", exist_ok=True)
    with open(out_txt, "w", encoding="utf-8") as w:
        w.write("# Alignment report (PairwiseAligner)\n")
        w.write(f"# Seq1: {id1}\n# Seq2: {id2}\n\n")
        w.write(a1 + "\n" + a2 + "\n\n")
        for k, v in m.items():
            w.write(f"{k}\t{v}\n")


def save_tsv(out_tsv: str, id1: str, id2: str, m: Dict[str, float]) -> None:
    os.makedirs(os.path.dirname(out_tsv) or ".", exist_ok=True)
    header = [
        "seq1", "seq2", "score", "length_aligned",
        "no_gaps_len", "matches", "identity", "gaps_seq1", "gaps_seq2"
    ]
    row = [
        id1, id2, m["score"], m["length_aligned"],
        m["no_gaps_len"], m["matches"], f"{m['identity']:.2f}",
        m["gaps_seq1"], m["gaps_seq2"]
    ]
    new = not os.path.exists(out_tsv)
    with open(out_tsv, "a", encoding="utf-8") as w:
        if new:
            w.write("\t".join(header) + "\n")
        w.write("\t".join(map(str, row)) + "\n")


def save_png(out_png: str, identity: float) -> None:
    os.makedirs(os.path.dirname(out_png) or ".", exist_ok=True)
    plt.figure(figsize=(4, 4))
    plt.bar(["Identity"], [identity], edgecolor="black")
    plt.ylim(0, 100)
    plt.ylabel("Percent (%)")
    plt.title("Alignment identity (PairwiseAligner)")
    plt.tight_layout()
    plt.savefig(out_png, dpi=200, bbox_inches="tight")
    plt.close()


# ---------- aligner config ----------

def make_aligner(method: str, scheme: str, gap_open: float, gap_extend: float) -> PairwiseAligner:
    """
    Создать и настроить PairwiseAligner:
      - method: 'global' или 'local'
      - scheme: 'blosum62' (подгружаем матрицу) или 'simple' (match/mismatch)
      - gap_open/extend: штрафы за разрывы (обычно отрицательные значения внутри алгоритма)
    """
    aligner = PairwiseAligner()
    aligner.mode = method  # 'global' или 'local'

    # Штрафы: в PairwiseAligner нужно ставить отрицательные значения (штрафы)
    aligner.open_gap_score = -abs(gap_open)
    aligner.extend_gap_score = -abs(gap_extend)

    if scheme.lower() == "blosum62":
        # Современная загрузка матрицы замен
        aligner.substitution_matrix = substitution_matrices.load("BLOSUM62")
    elif scheme.lower() == "simple":
        # Наивная схема без матрицы
        aligner.match_score = 1.0
        aligner.mismatch_score = -1.0
    else:
        raise ValueError("Unknown scheme (use 'blosum62' or 'simple')")

    return aligner


# ---------- cli ----------

def main():
    ap = argparse.ArgumentParser(description="Align two sequences using PairwiseAligner and report identity/score")
    ap.add_argument("-i1", required=True, help="FASTA 1")
    ap.add_argument("-i2", required=True, help="FASTA 2")
    ap.add_argument("-o", "--out", default="results/alignment.txt", help="output TXT")
    ap.add_argument("--tsv", default="results/alignment_summary.tsv", help="output TSV")
    ap.add_argument("--png", default="results/alignment_identity.png", help="output PNG")
    ap.add_argument("--method", choices=["global", "local"], default="global", help="alignment method")
    ap.add_argument("--matrix", choices=["blosum62", "simple"], default="blosum62", help="scoring scheme")
    ap.add_argument("--gap-open", type=float, default=10.0, help="gap opening penalty (positive number)")
    ap.add_argument("--gap-extend", type=float, default=0.5, help="gap extension penalty (positive number)")
    args = ap.parse_args()

    # Читаем последовательности
    id1, s1 = read_one_fasta(args.i1)
    id2, s2 = read_one_fasta(args.i2)

    # Настраиваем aligner
    aligner = make_aligner(args.method, args.matrix, args.gap_open, args.gap_extend)

    # Выравнивание (берём лучший)
    alignments = aligner.align(s1, s2)
    if len(alignments) == 0:
        raise RuntimeError("Alignment failed: empty result")
    best = alignments[0]

    # Восстанавливаем гаппированные строки и считаем метрики
    a1, a2 = build_gapped_strings(s1, s2, best)
    metrics = calc_metrics(a1, a2, best.score)

    # Сохраняем результаты
    save_txt(args.out, id1, id2, a1, a2, metrics)
    save_tsv(args.tsv, id1, id2, metrics)
    save_png(args.png, metrics["identity"])

    print(f"[OK] identity={metrics['identity']:.2f}% score={metrics['score']} -> {os.path.abspath(args.out)}")


if __name__ == "__main__":
    main()


C:\Users\Shaman\anaconda3\Lib\site-packages\Bio\pairwise2.py:278: BiopythonDeprecationWarning: Bio.pairwise2 has been deprecated, and we intend to remove it in a future release of Biopython. As an alternative, please consider using Bio.Align.PairwiseAligner as a replacement, and contact the Biopython developers if you still need the Bio.pairwise2 module.
  warnings.warn(
usage: ipykernel_launcher.py [-h] -i1 I1 -i2 I2 [-o OUT] [--tsv TSV] [--png PNG] [--method {global,local}]
                             [--matrix {blosum62,simple}] [--gap-open GAP_OPEN] [--gap-extend GAP_EXTEND]
ipykernel_launcher.py: error: the following arguments are required: -i1, -i2


SystemExit: 2